# Lecture 1: Probability
**Guided student notebook · about 90 minutes**

Draw samples, compare probability densities, and estimate a mean and covariance.
The three exercises follow the probability lecture: JAX (15 min), distributions
and moments (30 min), and a 2D Gaussian (35 min), with 10 min for discussion.

Run the cells in order. Replace `None` in the code marked **TODO**, and write short answers in the Markdown cells. The supplied displays show a reminder until the relevant exercise is complete.
Plotting code is supplied in [lecture_01_helpers.py](lecture_01_helpers.py).

## Setup
Complete installation before class. Install [uv](https://docs.astral.sh/uv/getting-started/installation/),
then run these commands in a **terminal**, starting in the course repository:

```bash
cd tutorials
uv venv --python 3.12
source .venv/bin/activate
uv pip install --python .venv -r requirements.txt
python -m jupyterlab
```

In Windows PowerShell, replace the activation line with `.venv\Scripts\Activate.ps1`.
Open this notebook with the environment's Python 3 kernel. In VS Code, select
`tutorials/.venv` as the kernel. Keep `requirements.txt` and `lecture_01_helpers.py`
in the same folder as the notebooks.

Use this environment for later tutorials. When requirements change, rerun the
`uv pip install` command from `tutorials/`, then restart the kernel.
[Environment instructions](https://docs.astral.sh/uv/pip/environments/).

In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import matplotlib.pyplot as plt

from lecture_01_helpers import (
    plot_distributions, report_moments, show_gaussian_sample,
)

jax.config.update("jax_enable_x64", True)
plt.style.use("default")

key = random.key(42)
setup_sample = random.normal(random.key(101), shape=(32,))
print(f"JAX {jax.__version__}; sample shape: {setup_sample.shape}")
fig, ax = plt.subplots()
ax.hist(setup_sample, bins=8, density=True)
ax.set(xlabel="x", ylabel="Density")
plt.show()

## 1. Arrays and random samples · 15 min
JAX is a Python library for numerical computing. Its `jax.numpy` interface
roughly mirrors NumPy, so many familiar array operations use the same syntax.
Compared with standard NumPy, JAX adds automatic differentiation,
just-in-time compilation for faster repeated computations, and support
for running array computations on GPUs and TPUs as well as CPUs.
Speedups depend on the workload; we use CPU JAX in this tutorial.

`jax.numpy` provides array operations; `jax.random` generates samples.
A JAX random key makes a draw reproducible. Calling the same sampling function
with the same key and arguments gives the same values, as this example deliberately shows.

References: [Getting started](https://docs.jax.dev/en/latest/beginner_guide.html)
and [random keys](https://docs.jax.dev/en/latest/random-numbers.html).

In [ ]:
values = jnp.array([1.0, 2.0, 3.0])
print("Array shape:", values.shape, " Mean:", float(jnp.mean(values)))

demo_key = random.key(7)
first_demo = random.normal(demo_key, shape=(5,))
repeated_demo = random.normal(demo_key, shape=(5,))  # Intentional key reuse.
print("First draw: ", first_demo)
print("Same key:   ", repeated_demo)
print("Identical?  ", bool(jnp.array_equal(first_demo, repeated_demo)))

**Task.** Draw two fresh batches of 1,000 standard-normal observations and
calculate the mean of the first batch. Use each supplied draw key once.
Rerun the cell to get new samples; rerunning the setup cell resets the master key.

A sampling shape of `(1000,)` means one array containing 1,000 observations.

In [ ]:
key, first_key, second_key = random.split(key, 3)

sample_a = None  # TODO: random.normal with first_key and shape=(1000,).
sample_b = None  # TODO: another batch, using second_key.
mean_a = None    # TODO: the sample mean of sample_a.

In [ ]:
if any(value is None for value in (sample_a, sample_b, mean_a)):
    print("Complete the three TODOs above, then rerun this cell.")
else:
    print("Shapes:", sample_a.shape, sample_b.shape)
    print("Identical?", bool(jnp.array_equal(sample_a, sample_b)))
    print(f"First sample mean: {float(mean_a):.4f}; population mean: 0")

**Explain.** Why did the worked example repeat exactly, while the split-key
samples differ? Should the sample mean equal zero exactly? What would restarting
the kernel and rerunning every cell reproduce?

**Your answer:**

## 2. Different PDFs, the same mean and variance · 30 min
These three distributions all have population mean $2$ and variance $1$.
Their parameters are supplied; your task is to draw and summarise samples.

| Distribution | Definition |
|---|---|
| Gaussian | $X\sim\mathcal N(2,1)$ |
| Uniform | $X\sim U(2-\sqrt3,\,2+\sqrt3)$ |
| Lognormal | $X=e^Y$, with $Y\sim\mathcal N(\mu_{\log},\sigma_{\log}^2)$ |

For the lognormal, $\sigma_{\log}^2=\log(1.25)$ and
$\mu_{\log}=\log 2-\tfrac12\log(1.25)$. These describe **$Y=\log X$**.

**Task.** Complete the three sampling expressions. For the lognormal, sample
a standard normal, scale and shift it to make $Y$, then exponentiate.

In [ ]:
population_mean = 2.0
population_variance = 1.0
population_sd = jnp.sqrt(population_variance)
lower = population_mean - jnp.sqrt(3.0) * population_sd
upper = population_mean + jnp.sqrt(3.0) * population_sd
log_variance = jnp.log(1.25)
log_sd = jnp.sqrt(log_variance)
log_mean = jnp.log(2.0) - log_variance / 2

In [ ]:
def draw_distributions(draw_key, n):
    gaussian_key, uniform_key, lognormal_key = random.split(draw_key, 3)
    gaussian = None   # TODO: shift and scale n standard-normal draws.
    uniform = None    # TODO: random.uniform(..., minval=lower, maxval=upper).
    lognormal = None  # TODO: jnp.exp(...) of the underlying Gaussian draws.
    return {"Gaussian": gaussian, "Uniform": uniform, "Lognormal": lognormal}

**Task.** Complete the sample mean and variance. Use the $n-1$ convention:

$$\bar x=\frac1n\sum_i x_i,\qquad s^2=\frac1{n-1}\sum_i(x_i-\bar x)^2.$$

You can use `jnp.mean` and `jnp.var(..., ddof=1)`. With `ddof=1`,
the variance calculation divides by $n-1$.

In [ ]:
def sample_moments(x):
    sample_mean = None      # TODO
    sample_variance = None  # TODO: use ddof=1.
    return sample_mean, sample_variance

In [ ]:
n = 400
key, draw_key = random.split(key)
samples = draw_distributions(draw_key, n)
report_moments(samples, sample_moments)
plot_distributions(samples)

**Compare.** The histograms are normalised as densities: each bar's area is
the fraction of observations in its bin. The curves labelled PDF are the population densities.

Run the larger-sample comparison below and inspect how the histograms change.
Both the numerical summaries and the plots use the same new samples.

In [ ]:
key, large_key = random.split(key)
large_samples = draw_distributions(large_key, 4000)
report_moments(large_samples, sample_moments)
plot_distributions(large_samples)

**Explain.**

1. What is the total area of a density-normalised histogram? Is a bar's height a probability?
2. Do equal population means and variances imply equal shapes, tails or interval probabilities?
3. Must every sample estimate improve when you increase the sample size?

**Your answer:**

## 3. Mean and covariance in two dimensions · 35 min
Draw observations $\mathbf X\sim\mathcal N_2(\boldsymbol\mu,\Sigma)$, where

$$\boldsymbol\mu=\begin{pmatrix}1\\-1\end{pmatrix},\qquad
\Sigma=\begin{pmatrix}\sigma_1^2&\rho\sigma_1\sigma_2\\
\rho\sigma_1\sigma_2&\sigma_2^2\end{pmatrix},\qquad
\sigma_1=1,\quad\sigma_2=1.5.$$

**Task.** Complete the draw using `random.multivariate_normal` with `shape=(n,)`.
The resulting array has shape `(n, 2)`: rows are observations and columns are components.
[Sampling reference](https://docs.jax.dev/en/latest/_autosummary/jax.random.multivariate_normal.html).

In [ ]:
mu = jnp.array([1.0, -1.0])
sigma_1, sigma_2 = 1.0, 1.5

def draw_gaussian_2d(draw_key, n, rho):
    Sigma = jnp.array([
        [sigma_1**2, rho * sigma_1 * sigma_2],
        [rho * sigma_1 * sigma_2, sigma_2**2],
    ])
    points = None  # TODO: draw from the Gaussian with mean mu and covariance Sigma.
    return points, Sigma

**Task.** Calculate the sample mean vector and covariance from centred samples:

$$\bar{\mathbf x}=\frac1n\sum_i\mathbf x_i,\qquad
S=\frac1{n-1}\sum_i(\mathbf x_i-\bar{\mathbf x})
(\mathbf x_i-\bar{\mathbf x})^{\mathsf T}.$$

Average over `axis=0`. Subtracting the two-component mean from every row gives
the centred array. If this array is called `centred`, its summed outer products
can be written `centred.T @ centred`. The `@` operator denotes matrix multiplication.

In [ ]:
def sample_mean_covariance(points):
    n = points.shape[0]
    estimated_mean = None        # TODO: mean over observations, axis=0.
    centred = None               # TODO: subtract the estimated mean from every row.
    estimated_covariance = None  # TODO: summed outer products, divided by n - 1.
    return estimated_mean, estimated_covariance

In [ ]:
n_2d = 250
rho = 0.75
key, draw_key = random.split(key)
points, Sigma = draw_gaussian_2d(draw_key, n_2d, rho)
estimates = sample_mean_covariance(points) if points is not None else (None, None)
show_gaussian_sample(points, mu, Sigma, *estimates)

**Predict, then run.** Keep the population mean and marginal variances fixed.
How should the cloud change at `rho = 0` and `rho = -0.75`?
Run the supplied comparison and check your prediction against both the plots
and the sample covariance matrices. Each case uses a fresh key.

In [ ]:
for rho_value in (0.0, -0.75):
    key, draw_key = random.split(key)
    new_points, new_Sigma = draw_gaussian_2d(draw_key, n_2d, rho_value)
    new_estimates = sample_mean_covariance(new_points) if new_points is not None else (None, None)
    show_gaussian_sample(new_points, mu, new_Sigma, *new_estimates)

**Repeat.** Return to positive correlation. The first case below is a fresh
dataset of the original size; the second has ten times as many observations.
Compare the sample means and covariances with the population values.

In [ ]:
for sample_size in (250, 2500):
    key, draw_key = random.split(key)
    repeat_points, repeat_Sigma = draw_gaussian_2d(draw_key, sample_size, 0.75)
    repeat_estimates = sample_mean_covariance(repeat_points) if repeat_points is not None else (None, None)
    show_gaussian_sample(repeat_points, mu, repeat_Sigma, *repeat_estimates, plot=False)

**Explain.**

1. What do the diagonal and off-diagonal covariance entries describe?
2. With zero population correlation, should the sample covariance be exactly zero?
3. Does zero covariance imply independence here? Does it do so for every distribution?
4. When you increase the sample size, should the cloud of individual observations become narrower?

**Your answer:**

## Discussion · 10 min
When you generate a fresh sample, what stays fixed and what changes?
What information do the mean and covariance retain about a distribution,
and what can they leave out?

**Your answer:**